In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import os, sys
from pathlib import Path

def find_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / '.git').exists():
            return p
    return start

root = find_root()
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print('Root:', root)

Root: /home/julian/Documents/chestx-classification


# Descarga de datos


In [2]:
import kagglehub

os.environ["KAGGLEHUB_CACHE"] = "."
path_sample_data = kagglehub.dataset_download("nih-chest-xrays/sample")
path_full_data = kagglehub.dataset_download("nih-chest-xrays/data")
print("Path to sample dataset files:", path_sample_data)
print("Path to full dataset files:", path_full_data)


/home/julian/Documents/chestx-classification/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to sample dataset files: ./datasets/nih-chest-xrays/sample/versions/4
Path to full dataset files: ./datasets/nih-chest-xrays/data/versions/3


# Análisis de datos

In [3]:
import pandas as pd

path_labels = os.path.join(path_full_data, 'Data_Entry_2017.csv')
labels_df = pd.read_csv(path_labels)
labels_df.head()

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN


# Preprocesamiento

### `PreprocessingConfigFactory`

In [ ]:
class PreprocessingConfigFactory:
    """Fábrica para configuraciones de preprocesamiento de datos."""
    
    def __init__(self, labels_df):
        """Inicializa la fábrica con el DataFrame de etiquetas."""
        self.labels_df = labels_df.copy()
    
    def get_available_configurations(self):
        """Retorna las configuraciones de preprocesamiento disponibles."""
        return {
            'example': self.config_example,
            'binary': self.config_binary,
            'effusion': self.config_effusion,
        }
    
    def get(self, config_name):
        """Retorna los DataFrames de train, val y test con la configuración especificada."""
        configs = self.get_available_configurations()
        if config_name not in configs:
            raise ValueError(f"Unknown config: {config_name}. Available: {list(configs.keys())}")
        return configs[config_name]()
    
    def config_example(self):
        """Configuración de ejemplo: convierte edad y hace split aleatorio."""
        self.labels_df['Patient Age'] = self.labels_df['Patient Age'].apply(self._convert_age)
        train_df, val_df, test_df = self._random_split(val_ratio=0.1, test_ratio=0.2, seed=0)
        return train_df, val_df, test_df
    
    def config_binary(self):
        """Configuración binaria para detección de Neumonía."""
        self.labels_df['Pneumonia_Label'] = self.labels_df['Finding Labels'].apply(
            lambda x: 1 if 'Pneumonia' in x else 0
        )
        train_df, val_df, test_df = self._patient_split(val_ratio=0.1, test_ratio=0.2, seed=0)
        return train_df, val_df, test_df
    
    def config_effusion(self):
        """Configuración para detección de Effusion."""
        self.labels_df['Effusion_Label'] = self.labels_df['Finding Labels'].apply(
            lambda x: 1 if 'Effusion' in x else 0
        )
        train_df, val_df, test_df = self._patient_split(val_ratio=0.1, test_ratio=0.2, seed=0)
        return train_df, val_df, test_df
    
    def _convert_age(self, age_str):
        """Convierte string de edad (Y/M/D) a años."""
        if isinstance(age_str, (int, float)):
            return float(age_str)
        if age_str.endswith('Y'):
            return int(age_str.replace('Y', ''))
        elif age_str.endswith('M'):
            return int(age_str.replace('M', '')) / 12
        elif age_str.endswith('D'):
            return int(age_str.replace('D', '')) / 365
        return int(age_str)
    
    def _random_split(self, val_ratio=0.1, test_ratio=0.2, seed=0):
        """Divide el DataFrame en train, val y test de forma aleatoria."""
        shuffled_df = self.labels_df.sample(frac=1, random_state=seed)
        val_idx = int(len(shuffled_df) * (1 - test_ratio - val_ratio))
        test_idx = int(len(shuffled_df) * (1 - test_ratio))

        train_df = shuffled_df.iloc[:val_idx]
        val_df = shuffled_df.iloc[val_idx:test_idx]
        test_df = shuffled_df.iloc[test_idx:]

        return train_df, val_df, test_df

    def _patient_split(self, val_ratio=0.1, test_ratio=0.2, seed=0):
        """Divide el DataFrame a nivel de paciente."""
        np.random.seed(seed)
        unique_patients = self.labels_df['Patient ID'].unique()
        np.random.shuffle(unique_patients)
        
        num_patients = len(unique_patients)
        val_idx = int(num_patients * (1 - test_ratio - val_ratio))
        test_idx = int(num_patients * (1 - test_ratio))
        
        train_patients = unique_patients[:val_idx]
        val_patients = unique_patients[val_idx:test_idx]
        test_patients = unique_patients[test_idx:]
        
        train_df = self.labels_df[self.labels_df['Patient ID'].isin(train_patients)]
        val_df = self.labels_df[self.labels_df['Patient ID'].isin(val_patients)]
        test_df = self.labels_df[self.labels_df['Patient ID'].isin(test_patients)]
        
        return train_df, val_df, test_df


# Datasets y Dataloaders

### `XRayBinaryDataset`

In [5]:
from torch.utils.data import Dataset
from PIL import Image

class XRayBinaryDataset(Dataset):
    """Dataset de radiografías de tórax con etiquetas binarias (Pneumonía)."""
    
    def __init__(self, labels_df, img_dir, transform=None):
        """Inicializa el dataset con DataFrame de etiquetas y directorio(es) de imágenes."""
        self.labels_df = labels_df
        self.img_dir = img_dir if isinstance(img_dir, list) else [img_dir]
        self.transform = transform
        
    def __len__(self):
        """Retorna el número de muestras en el dataset."""
        return len(self.labels_df)
    
    def __getitem__(self, idx):
        """Retorna la imagen y la etiqueta binaria para el índice dado."""
        img_name = self.labels_df.iloc[idx]['Image Index']
        img_path = None
        for directory in self.img_dir:
            path = os.path.join(directory, img_name)
            if os.path.exists(path):
                img_path = path
                break
        if img_path is None:
            raise FileNotFoundError(f"Imagen {img_name} no encontrada en ningún directorio")
        image = Image.open(img_path).convert('RGB')
        
        labels_str = self.labels_df.iloc[idx]['Finding Labels']
        label = 0
        if labels_str != 'No Finding':
            if 'Pneumonia' in labels_str.split('|'):
                label = 1
                
        label_tensor = torch.tensor([label], dtype=torch.float32)
        
        if self.transform:
            image = self.transform(image)
            
        return image, label_tensor

In [ ]:
class XRayEffusionDataset(Dataset):
    """Dataset de radiografías de tórax con etiquetas binarias (Effusion)."""
    
    def __init__(self, labels_df, img_dir, transform=None):
        """Inicializa el dataset con DataFrame de etiquetas y directorio(es) de imágenes."""
        self.labels_df = labels_df
        self.img_dir = img_dir if isinstance(img_dir, list) else [img_dir]
        self.transform = transform
        
    def __len__(self):
        """Retorna el número de muestras en el dataset."""
        return len(self.labels_df)
    
    def __getitem__(self, idx):
        """Retorna la imagen y la etiqueta binaria para el índice dado."""
        img_name = self.labels_df.iloc[idx]['Image Index']
        img_path = None
        for directory in self.img_dir:
            path = os.path.join(directory, img_name)
            if os.path.exists(path):
                img_path = path
                break
        if img_path is None:
            raise FileNotFoundError(f"Imagen {img_name} no encontrada en ningún directorio")
        image = Image.open(img_path).convert('RGB')
        
        labels_str = self.labels_df.iloc[idx]['Finding Labels']
        label = 0
        if labels_str != 'No Finding':
            if 'Effusion' in labels_str.split('|'):
                label = 1
                
        label_tensor = torch.tensor([label], dtype=torch.float32)
        
        if self.transform:
            image = self.transform(image)
            
        return image, label_tensor


### `DataLoaderFactory`

In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms

class DataLoaderFactory:
    """Fábrica para crear DataLoaders con diferentes configuraciones."""
    
    def __init__(self, labels_df, img_dir, num_workers=1, is_train=False):
        """Inicializa la fábrica con el DataFrame de etiquetas y directorio(s) de imágenes."""
        self.labels_df = labels_df
        self.img_dir = img_dir if isinstance(img_dir, list) else [img_dir]
        self.num_workers = num_workers
        self.is_train = is_train
    
    def get_available_configurations(self):
        """Retorna las configuraciones de DataLoader disponibles."""
        return {
            'binary': self.create_binary,
            'binary_low_res': self.create_binary_low_res,
            'effusion': self.create_effusion,
            'effusion_low_res': self.create_effusion_low_res
        }

    def get(self, config_name):
        """Retorna un DataLoader con la configuración especificada."""
        configs = self.get_available_configurations()
        if config_name not in configs:
            raise ValueError(f"Unknown config: {config_name}. Available: {list(configs.keys())}")
        return configs[config_name]()

    def create_binary(self):
        """Crea un DataLoader binario con resize, normalize y random flip (si es train)."""
        transform_list = [
            transforms.Resize((224, 224))
        ]
        if self.is_train:
            transform_list.append(transforms.RandomHorizontalFlip())
            
        transform_list.extend([
            transforms.ToTensor(),
            # normalizacion usada en densenet121
            # https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.densenet121.html
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        transform = transforms.Compose(transform_list)
        dataset = XRayBinaryDataset(self.labels_df, self.img_dir, transform=transform)
        return DataLoader(dataset, batch_size=16, shuffle=self.is_train, num_workers=self.num_workers)
 
    def create_binary_low_res(self):
        """Crea un DataLoader binario con resize 128x128, normalize y random flip (si es train)."""
        transform_list = [
            transforms.Resize((128, 128))
        ]
        if self.is_train:
            transform_list.append(transforms.RandomHorizontalFlip())
            
        transform_list.extend([
            transforms.ToTensor(),
            # normalizacion usada en densenet121
            # https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.densenet121.html
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        transform = transforms.Compose(transform_list)
        dataset = XRayBinaryDataset(self.labels_df, self.img_dir, transform=transform)
        return DataLoader(dataset, batch_size=16, shuffle=self.is_train, num_workers=self.num_workers)
    
    def create_effusion(self):
        transform_list = [
            transforms.Resize((224, 224))
        ]
        if self.is_train:
            transform_list.append(transforms.RandomHorizontalFlip())
            
        transform_list.extend([
            transforms.ToTensor(),
            # normalizacion usada en densenet121
            # https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.densenet121.html
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        transform = transforms.Compose(transform_list)
        dataset = XRayEffusionDataset(self.labels_df, self.img_dir, transform=transform)
        return DataLoader(dataset, batch_size=16, shuffle=self.is_train, num_workers=self.num_workers)
    
    def create_effusion_low_res(self):
        transform_list = [
            transforms.Resize((128, 128))
        ]
        if self.is_train:
            transform_list.append(transforms.RandomHorizontalFlip())
            
        transform_list.extend([
            transforms.ToTensor(),
            # normalizacion usada en densenet121
            # https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.densenet121.html
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        transform = transforms.Compose(transform_list)
        dataset = XRayEffusionDataset(self.labels_df, self.img_dir, transform=transform)
        return DataLoader(dataset, batch_size=16, shuffle=self.is_train, num_workers=self.num_workers)

# Modelos

### `BaseModel`

In [7]:
import torch.nn as nn
import torch.optim as optim
from abc import ABC
from tqdm import tqdm

class BaseModel(nn.Module, ABC):
    """Clase base abstracta para modelos con método de entrenamiento."""
    
    def train_model(self, dataloader, val_dataloader=None, lr=0.001, device=None, num_epochs=10):
        """Entrena el modelo con el dataloader proporcionado."""
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        self.optimizer = optim.Adam(self.parameters(), lr=lr)
        self.to(device)
        
        for epoch in range(num_epochs):
            self.train()
            running_loss = 0.0
            
            for images, labels in tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]"):
                images = images.to(device)
                labels = labels.to(device)
                
                outputs = self(images)
                loss = self.criterion(outputs, labels)
                
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
                
                running_loss += loss.item()
            
            epoch_loss = running_loss / len(dataloader)
            print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {epoch_loss:.4f}")
            
            if val_dataloader is not None:
                self.eval()
                val_loss = 0.0
                with torch.no_grad():
                    for images, labels in tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"):
                        images = images.to(device)
                        labels = labels.to(device)
                        outputs = self(images)
                        loss = self.criterion(outputs, labels)
                        val_loss += loss.item()
                epoch_val_loss = val_loss / len(val_dataloader)
                print(f"Epoch {epoch+1}/{num_epochs}, Val Loss: {epoch_val_loss:.4f}")
        
        return self

### `DenseNetBinary`

In [10]:
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as torchvision_models


class DenseNetBinary(BaseModel):
    """DenseNet-121 para clasificación binaria de Neumonía."""
    
    def __init__(self):
        super(DenseNetBinary, self).__init__()
        
        # cargamos el modelo de DenseNet-121 preentrenado
        # se reemplaza el clasificador para una tarea binaria
        self.model = torchvision_models.densenet121(weights=torchvision_models.DenseNet121_Weights.DEFAULT)
        num_ftrs = self.model.classifier.in_features
        self.model.classifier = nn.Linear(num_ftrs, 1)
        self.criterion = None
        self.optimizer = None
        
    def forward(self, x):
        return self.model(x)
        
    def train_model(self, dataloader, val_dataloader, lr=0.001, device=None, num_epochs=1):
        """Sobrescribe train_model para calcular pesos, usar Weighted BCE y scheduler."""
        import copy
        
        # calculo de pesos de clases a partir del dataloader de entrenamiento
        # para tener en cuenta el desbalance de clases 
        print("Calculando pesos de clases (w_pos, w_neg) a partir del dataloader...")
        num_pos = 0
        num_neg = 0
        for _, labels in dataloader:
            pos = labels.sum().item()
            num_pos += pos
            num_neg += (labels.shape[0] - pos)
            
        total = num_pos + num_neg
        w_pos = num_neg / total if total > 0 else 0
        w_neg = num_pos / total if total > 0 else 0
        print(f"Pesos calculados: w_pos={w_pos:.4f}, w_neg={w_neg:.4f}")
        
        # setear device para transferir el modelo 
        # convertir pesos de clases a tensores y enviarlos a device
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Device: {device}")
        w_pos_tensor = torch.tensor(w_pos, dtype=torch.float32).to(device)
        w_neg_tensor = torch.tensor(w_neg, dtype=torch.float32).to(device)
        
        # funcion de loss tipo Weighted BCE:
        # loss = BCE(y, y') * (w_pos * y + w_neg * (1-y)) 
        def weighted_loss(logits, targets):
            bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
            weight_matrix = targets * w_pos_tensor + (1 - targets) * w_neg_tensor
            return (bce * weight_matrix).mean()
        self.criterion = weighted_loss
        
        # se usa un optimizador Adam con un learning rate inicial
        # y un scheduler para reducir el learning rate en un factor 
        # de 0.1 cuando la validación no mejora por 2 épocas seguidas
        self.optimizer = optim.Adam(self.parameters(), lr=lr)
        self.to(device)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.1, patience=2
        )
        
        # se guardara solo el modelo con mejor validation loss
        # ademas listas con loss de train y validation por epoca
        best_val_loss = float('inf')
        best_model_wts = copy.deepcopy(self.state_dict())
        self.train_losses = []
        self.val_losses = []

        # bucle de entrenamiento
        for epoch in range(num_epochs):
            self.train()
            running_loss = 0.0
            for images, labels in tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]"):
                images = images.to(device)
                labels = labels.to(device)
                
                # forward pass
                outputs = self(images)
                loss = self.criterion(outputs, labels)
                
                # backward pass
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
                
                running_loss += loss.item()
            
            epoch_loss = running_loss / len(dataloader)
            print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {epoch_loss:.4f}")
            self.train_losses.append(epoch_loss)
            
            # fase de validacion
            # evaluamos el modelo en el dataloader de validacion
            # y calculamos la perdida
            self.eval()
            val_loss = 0.0
            with torch.no_grad():
                for images, labels in tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"):
                    images = images.to(device)
                    labels = labels.to(device)
                    outputs = self(images)
                    loss = self.criterion(outputs, labels)
                    val_loss += loss.item()
                    
            epoch_val_loss = val_loss / len(val_dataloader)
            print(f"Epoch {epoch+1}/{num_epochs}, Val Loss: {epoch_val_loss:.4f}")
            self.val_losses.append(epoch_val_loss)
            scheduler.step(epoch_val_loss)

            # guardar el modelo con mejor validation loss
            if epoch_val_loss < best_val_loss:
                print("Mejoró la pérdida de validación, guardando el modelo...")
                best_val_loss = epoch_val_loss
                best_model_wts = copy.deepcopy(self.state_dict())
            else:
                print("Pérdida de validación no mejoró...")

        print(f"Training complete. Best val loss: {best_val_loss:.4f}")
        self.load_state_dict(best_model_wts)
        
        
    def generate_cam(self, x):
        # x es un batch de imágenes [Batch_Size, 3, H, W]
        # devuelve [Batch_Size, H_feat, W_feat]
        features = self.model.features(x)
        features = F.relu(features, inplace=False)
        weights = self.model.classifier.weight[0]
        cam = torch.einsum('c,bchw->bhw', weights, features)
        return cam

### `ModelFactory`

In [11]:
class ModelFactory:
    """Fábrica para crear modelos con diferentes configuraciones."""
    
    def __init__(self):
        pass
    
    def get_available_configurations(self):
        """Retorna las configuraciones de modelo disponibles."""
        return {
            'densenet_binary': self.create_densenet_binary,
        }

    def get(self, model_name):
        """Retorna un modelo con la configuración especificada."""
        configs = self.get_available_configurations()
        if model_name not in configs:
            raise ValueError(f"Unknown model name: {model_name}. Available: {list(configs.keys())}")
        return configs[model_name]()
    
    def create_densenet_binary(self):
        """Crea un modelo DenseNet-121 para clasificación binaria."""
        return DenseNetBinary()

# Utilidades

### `save_model` `load_model` `load_test_dataloader` 


In [12]:
def save_model(
    filepath,
    model, 
    preprocessing_name,
    dataloader_name, 
    model_name,
    labels_dir,
    img_dir,
    train_losses=None,
    val_losses=None,
):
    """Guarda el modelo, su configuración y opcionalmente las listas de pérdidas de entrenamiento y validación."""
    save_dict = {
        'state_dict': model.state_dict(),
        'preprocessing_name': preprocessing_name,
        'dataloader_name': dataloader_name,
        'model_name': model_name,
        'labels_dir': labels_dir,
        'img_dir': img_dir,
        'train_losses': train_losses,
        'val_losses': val_losses,
    }
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    torch.save(save_dict, filepath)
    print(f"Modelo '{model_name}' guardado en {filepath}")


def load_model(filepath):
    """Carga un modelo desde un archivo guardado."""
    save_dict = torch.load(filepath)
    model_factory = ModelFactory()
    model = model_factory.get(save_dict['model_name'])
    model.load_state_dict(save_dict['state_dict'])
    return model

def load_test_dataloader(filepath):
    """Carga el DataLoader de test desde un archivo de modelo guardado."""
    save_dict = torch.load(filepath)
    labels_df = pd.read_csv(save_dict['labels_dir'])
    pp_factory = PreprocessingConfigFactory(labels_df)
    _, _, test_df = pp_factory.get(save_dict['preprocessing_name'])
    dl_factory = DataLoaderFactory(test_df, save_dict['img_dir'], is_train=False)
    return dl_factory.get(save_dict['dataloader_name'])

# Métodos de diagnóstico

In [13]:
from sklearn.metrics import roc_curve, auc

def plot_roc_curve(model, dataloader, device=None):
    """Genera la curva ROC y devuelve la figura."""
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    model.to(device)
    all_labels = []
    all_probs = []
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Calculating Probabilities"):
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('Receiver Operating Characteristic')
    ax.legend(loc="lower right")
    return fig

In [14]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

def plot_confusion_matrix_binary(model, dataloader, threshold=0.5, device=None):
    """Genera la matriz de confusion para un modelo binario y devuelve la figura."""
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    model.to(device)
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Calculating Predictions"):
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs).cpu().numpy()
            
            preds = (probs >= threshold).astype(int)
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())    
    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title('Confusion Matrix')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    return fig

In [15]:
def plot_cam_overlay(model, image_tensor, original_image=None, device=None):
    """Genera el CAM para una imagen y lo superpone usando Matplotlib."""
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    model.to(device)
    image_tensor = image_tensor.to(device)
    with torch.no_grad():
        cam = model.generate_cam(image_tensor)
    cam = cam.cpu().numpy()[0] # [H, W]
    cam = cam - np.min(cam)
    cam = cam / (np.max(cam) + 1e-8)
    cam_tensor = torch.from_numpy(cam).unsqueeze(0).unsqueeze(0) # [1, 1, H, W]
    if original_image is not None:
        h, w = np.array(original_image).shape[:2]
    else:
        h, w = image_tensor.shape[2], image_tensor.shape[3]
    cam_resized = F.interpolate(cam_tensor, size=(h, w), mode='bilinear', align_corners=False)
    cam_resized = cam_resized.squeeze().numpy()
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    if original_image is not None:
        img_show = original_image
        axes[0].imshow(img_show, cmap='gray')
        axes[1].imshow(img_show, cmap='gray')
    else:
        img_show = image_tensor.cpu().squeeze().numpy()
        img_show = np.transpose(img_show, (1, 2, 0))
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img_show = std * img_show + mean
        img_show = np.clip(img_show, 0, 1)
        img_gray = np.dot(img_show[...,:3], [0.2989, 0.5870, 0.1140])
        axes[0].imshow(img_gray, cmap='gray')
        axes[1].imshow(img_gray, cmap='gray')
        
    im = axes[1].imshow(cam_resized, cmap='jet', alpha=0.5)
    axes[0].axis('off')
    axes[0].set_title("Input")
    axes[1].axis('off')
    axes[1].set_title("CAM")
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    plt.tight_layout()
    return fig

# Ejecutables

In [16]:
def train_model(
    labels_dir, 
    img_dir, 
    preprocessing_name, 
    dataloader_name, 
    model_name, 
    output_path,
    num_epochs=10, 
    lr=0.001, 
    num_workers=1, 
):
    """Entrena un modelo con las configuraciones especificadas y lo guarda."""
    labels_df = pd.read_csv(labels_dir)
    pp_factory = PreprocessingConfigFactory(labels_df)

    train_df, val_df, test_df = pp_factory.get(preprocessing_name)
    print(f"Entrenando con {len(train_df)} muestras, validando con {len(val_df)} muestras")

    dl_factory_train = DataLoaderFactory(train_df, img_dir, num_workers=num_workers, is_train=True)
    train_dataloader = dl_factory_train.get(dataloader_name)

    dl_factory_val = DataLoaderFactory(val_df, img_dir, num_workers=num_workers, is_train=False)
    val_dataloader = dl_factory_val.get(dataloader_name)

    model_factory = ModelFactory()
    model = model_factory.get(model_name)
    model.train_model(train_dataloader, val_dataloader=val_dataloader, num_epochs=num_epochs, lr=lr)

    save_model(
        output_path, model, preprocessing_name, 
        dataloader_name, model_name, labels_dir, 
        img_dir, train_losses=model.train_losses, 
        val_losses=model.val_losses
    )
    return 

In [17]:
train_model(
    labels_dir='datasets/nih-chest-xrays/data/versions/3/Data_Entry_2017.csv',
    img_dir=[
        'datasets/nih-chest-xrays/data/versions/3/images_001/images',
        'datasets/nih-chest-xrays/data/versions/3/images_002/images',
        'datasets/nih-chest-xrays/data/versions/3/images_003/images',
        'datasets/nih-chest-xrays/data/versions/3/images_004/images',
        'datasets/nih-chest-xrays/data/versions/3/images_005/images',
        'datasets/nih-chest-xrays/data/versions/3/images_006/images',
        'datasets/nih-chest-xrays/data/versions/3/images_007/images',
        'datasets/nih-chest-xrays/data/versions/3/images_008/images',
        'datasets/nih-chest-xrays/data/versions/3/images_009/images',
        'datasets/nih-chest-xrays/data/versions/3/images_010/images',
        'datasets/nih-chest-xrays/data/versions/3/images_011/images',
        'datasets/nih-chest-xrays/data/versions/3/images_012/images',
    ],
    preprocessing_name='binary',
    dataloader_name='binary_low_res',
    model_name='densenet_binary',
    output_path='results/models/full_data_binary_binary_low_res_densenet_binary_1epoch.pth',
    num_epochs=1,
    lr=0.001,
    num_workers=11,
)

train_model(
    labels_dir='datasets/nih-chest-xrays/data/versions/3/Data_Entry_2017.csv',
    img_dir=[
        'datasets/nih-chest-xrays/data/versions/3/images_001/images',
        'datasets/nih-chest-xrays/data/versions/3/images_002/images',
        'datasets/nih-chest-xrays/data/versions/3/images_003/images',
        'datasets/nih-chest-xrays/data/versions/3/images_004/images',
        'datasets/nih-chest-xrays/data/versions/3/images_005/images',
        'datasets/nih-chest-xrays/data/versions/3/images_006/images',
        'datasets/nih-chest-xrays/data/versions/3/images_007/images',
        'datasets/nih-chest-xrays/data/versions/3/images_008/images',
        'datasets/nih-chest-xrays/data/versions/3/images_009/images',
        'datasets/nih-chest-xrays/data/versions/3/images_010/images',
        'datasets/nih-chest-xrays/data/versions/3/images_011/images',
        'datasets/nih-chest-xrays/data/versions/3/images_012/images',
    ],
    preprocessing_name='binary',
    dataloader_name='binary_low_res',
    model_name='densenet_binary',
    output_path='results/models/full_data_binary_binary_low_res_densenet_binary_15epochs.pth',
    num_epochs=15,
    lr=0.001,
    num_workers=11,
)

train_model(
    labels_dir='datasets/nih-chest-xrays/data/versions/3/Data_Entry_2017.csv',
    img_dir=[
        'datasets/nih-chest-xrays/data/versions/3/images_001/images',
        'datasets/nih-chest-xrays/data/versions/3/images_002/images',
        'datasets/nih-chest-xrays/data/versions/3/images_003/images',
        'datasets/nih-chest-xrays/data/versions/3/images_004/images',
        'datasets/nih-chest-xrays/data/versions/3/images_005/images',
        'datasets/nih-chest-xrays/data/versions/3/images_006/images',
        'datasets/nih-chest-xrays/data/versions/3/images_007/images',
        'datasets/nih-chest-xrays/data/versions/3/images_008/images',
        'datasets/nih-chest-xrays/data/versions/3/images_009/images',
        'datasets/nih-chest-xrays/data/versions/3/images_010/images',
        'datasets/nih-chest-xrays/data/versions/3/images_011/images',
        'datasets/nih-chest-xrays/data/versions/3/images_012/images',
    ],
    preprocessing_name='binary',
    dataloader_name='binary',
    model_name='densenet_binary',
    output_path='results/models/full_data_binary_binary_densenet_binary_1epoch.pth',
    num_epochs=1,
    lr=0.001,
    num_workers=11,
)

train_model(
    labels_dir='datasets/nih-chest-xrays/data/versions/3/Data_Entry_2017.csv',
    img_dir=[
        'datasets/nih-chest-xrays/data/versions/3/images_001/images',
        'datasets/nih-chest-xrays/data/versions/3/images_002/images',
        'datasets/nih-chest-xrays/data/versions/3/images_003/images',
        'datasets/nih-chest-xrays/data/versions/3/images_004/images',
        'datasets/nih-chest-xrays/data/versions/3/images_005/images',
        'datasets/nih-chest-xrays/data/versions/3/images_006/images',
        'datasets/nih-chest-xrays/data/versions/3/images_007/images',
        'datasets/nih-chest-xrays/data/versions/3/images_008/images',
        'datasets/nih-chest-xrays/data/versions/3/images_009/images',
        'datasets/nih-chest-xrays/data/versions/3/images_010/images',
        'datasets/nih-chest-xrays/data/versions/3/images_011/images',
        'datasets/nih-chest-xrays/data/versions/3/images_012/images',
    ],
    preprocessing_name='binary',
    dataloader_name='binary',
    model_name='densenet_binary',
    output_path='results/models/full_data_binary_binary_densenet_binary_15epochs.pth',
    num_epochs=15,
    lr=0.001,
    num_workers=11,
)

Entrenando con 78459 muestras, validando con 11263 muestras
Calculando pesos de clases (w_pos, w_neg) a partir del dataloader...


KeyboardInterrupt: 

In [18]:
def plot_confusion_and_save(model_path, output_path, threshold=0.5):
    """Plots the confusion matrix and saves it to a file."""
    if not os.path.exists(model_path):
        print(f"Error: {model_path} no encontrado")
        sys.exit(1)
    model = load_model(model_path)
    dataloader_test = load_test_dataloader(model_path)
    fig = plot_confusion_matrix_binary(model, dataloader_test, threshold=threshold)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    fig.savefig(output_path)
    print(f"Matriz de confusión guardada en: {output_path}")

In [ ]:
plot_confusion_and_save(
    model_path='results/models/full_data_binary_binary_low_res_densenet_binary_1epoch.pth', 
    output_path='results/confusion/full_data_binary_binary_low_res_densenet_binary_1epoch.pdf'
)
plot_confusion_and_save(
    model_path='results/models/full_data_binary_binary_low_res_densenet_binary_15epochs.pth', 
    output_path='results/confusion/full_data_binary_binary_low_res_densenet_binary_15epochs.pdf'
)
plot_confusion_and_save(
    model_path='results/models/full_data_binary_binary_densenet_binary_1epoch.pth', 
    output_path='results/confusion/full_data_binary_binary_densenet_binary_1epoch.pdf'
)
plot_confusion_and_save(
    model_path='results/models/full_data_binary_binary_densenet_binary_15epochs.pth', 
    output_path='results/confusion/full_data_binary_binary_densenet_binary_15epochs.pdf'
)

In [ ]:
def plot_roc_and_save(model_path, output_path):
    """Plots the ROC curve and saves it to a file."""
    if not os.path.exists(model_path):
        print(f"Error: {model_path} no encontrado")
        sys.exit(1)
    model = load_model(model_path)
    dataloader_test = load_test_dataloader(model_path)
    fig = plot_roc_curve(model, dataloader_test)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    fig.savefig(output_path)
    print(f"Curva ROC guardada en: {output_path}")

In [ ]:
plot_roc_and_save(
    model_path='results/models/full_data_binary_binary_low_res_densenet_binary_1epoch.pth',
    output_path='results/roc/full_data_binary_binary_low_res_densenet_binary_1epoch.pdf'
)
plot_roc_and_save(
    model_path='results/models/full_data_binary_binary_low_res_densenet_binary_15epochs.pth',
    output_path='results/roc/full_data_binary_binary_low_res_densenet_binary_15epochs.pdf'
)
plot_roc_and_save(
    model_path='results/models/full_data_binary_binary_densenet_binary_1epoch.pth',
    output_path='results/roc/full_data_binary_binary_densenet_binary_1epoch.pdf'
)
plot_roc_and_save(
    model_path='results/models/full_data_binary_binary_densenet_binary_15epochs.pth',
    output_path='results/roc/full_data_binary_binary_densenet_binary_15epochs.pdf'
)

In [19]:
torch.multiprocessing.set_sharing_strategy('file_system')
import random

def generate_batch_cams(
    model_path, 
    num_images=16, 
    mode='random', 
    output_dir='results/cam', 
    output_prefix='cam', 
    seed=42
    ):
    if not os.path.exists(model_path):
        print(f"Error: {model_path} no encontrado")
        sys.exit(1)
    model = load_model(model_path)
    dataloader_test = load_test_dataloader(model_path)
    dataset = dataloader_test.dataset
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    
    os.makedirs(output_dir, exist_ok=True)
    random.seed(seed)
    
    selected_indices = []
    
    if mode == 'random':
        print(f"Seleccionando {num_images} imágenes al azar...")
        selected_indices = random.sample(range(len(dataset)), min(num_images, len(dataset)))
    else:
        print(f"Buscando {num_images} imágenes con condición '{mode.upper()}'...")
        search_indices = list(range(len(dataset)))
        random.shuffle(search_indices)
        
        for idx in search_indices:
            img, lbl = dataset[idx]
            img_tensor = img.unsqueeze(0).to(device)
            true_label = int(lbl.item())
            
            with torch.no_grad():
                output = model(img_tensor)
                prob = torch.sigmoid(output).item()
                pred_label = 1 if prob >= 0.5 else 0
                
            if mode == 'tp' and true_label == 1 and pred_label == 1:
                selected_indices.append(idx)
            elif mode == 'fp' and true_label == 0 and pred_label == 1:
                selected_indices.append(idx)
            elif mode == 'fn' and true_label == 1 and pred_label == 0:
                selected_indices.append(idx)
            elif mode == 'tn' and true_label == 0 and pred_label == 0:
                selected_indices.append(idx)
                
            if len(selected_indices) >= num_images:
                break
                
        print(f"Se encontraron {len(selected_indices)} imágenes de tipo '{mode.upper()}'.")
        if len(selected_indices) == 0:
            print("No se encontraron imágenes que cumplan la condición. Terminando.")
            return

    print(f"Generando CAMs...")
    
    for idx in selected_indices:
        image_tensor, lbl = dataset[idx]
        image_tensor_batch = image_tensor.unsqueeze(0).to(device)
        true_label = int(lbl.item())
        img_name = dataset.labels_df.iloc[idx]['Image Index']
        with torch.no_grad():
            output = model(image_tensor_batch)
            prob = torch.sigmoid(output).item()
            
        pred_label = 1 if prob >= 0.5 else 0
        
        if true_label == 1 and pred_label == 1:
            category = "True Positive"
        elif true_label == 0 and pred_label == 0:
            category = "True Negative"
        elif true_label == 0 and pred_label == 1:
            category = "False Positive"
        else:
            category = "False Negative"
            
        print(f"Procesando {img_name}: {category} (Prob: {prob:.4f})")
        fig = plot_cam_overlay(model, image_tensor_batch)
        fig.suptitle(f"CAM: {category} ({prob:.2f} prob)\nImage: {img_name}", fontsize=14)
        plt.tight_layout()
        base_name = os.path.splitext(img_name)[0]
        mode_prefix = f"{mode}_" if mode != 'random' else ""
        output_path = os.path.join(output_dir, f"{output_prefix}_{mode_prefix}{base_name}.pdf")
        fig.savefig(output_path)
        plt.close(fig) 
        
    print(f"Todas las imágenes han sido guardadas en {output_dir}/")
    return


In [ ]:
# full_data_binary_binary_low_res_densenet_binary_1epoch
generate_batch_cams(
    model_path='results/models/full_data_binary_binary_low_res_densenet_binary_1epoch.pth', 
    num_images=10, 
    mode='tp', 
    output_dir='results/cam', 
    output_prefix='full_data_binary_binary_low_res_densenet_binary_1epoch',
    seed=1
)
generate_batch_cams(
    model_path='results/models/full_data_binary_binary_low_res_densenet_binary_1epoch.pth', 
    num_images=10, 
    mode='fp', 
    output_dir='results/cam', 
    output_prefix='full_data_binary_binary_low_res_densenet_binary_1epoch',
    seed=1
)
generate_batch_cams(
    model_path='results/models/full_data_binary_binary_low_res_densenet_binary_1epoch.pth', 
    num_images=10, 
    mode='fn', 
    output_dir='results/cam', 
    output_prefix='full_data_binary_binary_low_res_densenet_binary_1epoch',
    seed=1
)
generate_batch_cams(
    model_path='results/models/full_data_binary_binary_low_res_densenet_binary_1epoch.pth', 
    num_images=10, 
    mode='tn', 
    output_dir='results/cam', 
    output_prefix='full_data_binary_binary_low_res_densenet_binary_1epoch',
    seed=1
)

# full_data_binary_binary_low_res_densenet_binary_15epochs
generate_batch_cams(
    model_path='results/models/full_data_binary_binary_low_res_densenet_binary_15epochs.pth', 
    num_images=10, 
    mode='tp', 
    output_dir='results/cam', 
    output_prefix='full_data_binary_binary_low_res_densenet_binary_15epochs',
    seed=1
)
generate_batch_cams(
    model_path='results/models/full_data_binary_binary_low_res_densenet_binary_15epochs.pth', 
    num_images=10, 
    mode='fp', 
    output_dir='results/cam', 
    output_prefix='full_data_binary_binary_low_res_densenet_binary_15epochs',
    seed=1
)
generate_batch_cams(
    model_path='results/models/full_data_binary_binary_low_res_densenet_binary_15epochs.pth', 
    num_images=10, 
    mode='fn', 
    output_dir='results/cam', 
    output_prefix='full_data_binary_binary_low_res_densenet_binary_15epochs',
    seed=1
)
generate_batch_cams(
    model_path='results/models/full_data_binary_binary_low_res_densenet_binary_15epochs.pth', 
    num_images=10, 
    mode='tn', 
    output_dir='results/cam', 
    output_prefix='full_data_binary_binary_low_res_densenet_binary_15epochs',
    seed=1
)

# full_data_binary_binary_densenet_binary_1epoch
generate_batch_cams(
    model_path='results/models/full_data_binary_binary_densenet_binary_1epoch.pth', 
    num_images=10, 
    mode='tp', 
    output_dir='results/cam', 
    output_prefix='full_data_binary_binary_densenet_binary_1epoch',
    seed=1
)
generate_batch_cams(
    model_path='results/models/full_data_binary_binary_densenet_binary_1epoch.pth', 
    num_images=10, 
    mode='fp', 
    output_dir='results/cam', 
    output_prefix='full_data_binary_binary_densenet_binary_1epoch',
    seed=1
)
generate_batch_cams(
    model_path='results/models/full_data_binary_binary_densenet_binary_1epoch.pth', 
    num_images=10, 
    mode='fn', 
    output_dir='results/cam', 
    output_prefix='full_data_binary_binary_densenet_binary_1epoch',
    seed=1
)
generate_batch_cams(
    model_path='results/models/full_data_binary_binary_densenet_binary_1epoch.pth', 
    num_images=10, 
    mode='tn', 
    output_dir='results/cam', 
    output_prefix='full_data_binary_binary_densenet_binary_1epoch',
    seed=1
)

# full_data_binary_binary_densenet_binary_15epochs
generate_batch_cams(
    model_path='results/models/full_data_binary_binary_densenet_binary_15epochs.pth', 
    num_images=10, 
    mode='tp', 
    output_dir='results/cam', 
    output_prefix='full_data_binary_binary_densenet_binary_15epochs',
    seed=1
)
generate_batch_cams(
    model_path='results/models/full_data_binary_binary_densenet_binary_15epochs.pth', 
    num_images=10, 
    mode='fp', 
    output_dir='results/cam', 
    output_prefix='full_data_binary_binary_densenet_binary_15epochs',
    seed=1
)
generate_batch_cams(
    model_path='results/models/full_data_binary_binary_densenet_binary_15epochs.pth', 
    num_images=10, 
    mode='fn', 
    output_dir='results/cam', 
    output_prefix='full_data_binary_binary_densenet_binary_15epochs',
    seed=1
)
generate_batch_cams(
    model_path='results/models/full_data_binary_binary_densenet_binary_15epochs.pth', 
    num_images=10, 
    mode='tn', 
    output_dir='results/cam', 
    output_prefix='full_data_binary_binary_densenet_binary_15epochs',
    seed=1
)

# Resultados